# Supplementary Table S11: Fixed-effect multinomial logit regression after removing country self-citations

This notebook fits the multinomial logit models (1)-(6) with temporal fixed-effects of Supplementary Table S6 on the national citation preferences recomputed after removing country self-citations (citations in which the citing and cited articles share a country; Supplementary Note 8), and prints the LaTeX regression table.

It uses the model code of **9_GlobalCitationNetwork_TableS6_multinomial**; only the input differs from notebook 9.

In [1]:
import os
from itertools import chain
import pandas as pd
import numpy as np

import scipy.stats as spstats
from statsmodels.stats.multitest import multipletests as holm
import statsmodels.api as sm

from vizutils import make_multinomial_latex_table, make_multinomial_latex_tables_split
from sourcedatautils import write_source_data, path2sourcedata, path2data

# Citation preferences without country self-citations

In [2]:
country_auc_df = pd.read_csv(os.path.join(path2data, 'oa_countrycites_noselfauthoraff_noselfcite_auc.csv.gz'))

country_auc_df = country_auc_df[country_auc_df['N']>=50]

country_auc_df['Zscore'] = (country_auc_df['AUC'] - 0.5)/np.sqrt(country_auc_df['Cov'])
# one-sided test in the direction of the observed preference: H1 is AUC > 0.5 (over-citation)
# when Z > 0, and AUC < 0.5 (under-citation) when Z < 0
country_auc_df['pvalue'] = spstats.norm.sf(np.abs(country_auc_df['Zscore'].values))

def reject_holm(pvalues):
    reject, pvalues, _, _ = holm(pvalues, alpha=0.01, method='hs', is_sorted=False, returnsorted=False)
    return reject
country_auc_df['Significant'] = country_auc_df.groupby('CitedYear')['pvalue'].transform(reject_holm)

net_auc = country_auc_df[country_auc_df['CitingCountry'] != country_auc_df['CitedCountry']].sort_values(by=['CitedYear', 'CitedCountry']).reset_index(drop=True)
net_auc = net_auc[net_auc['Significant']]

syear, eyear = 1990, 2017
net_auc = net_auc[net_auc['CitedYear'] >= syear]
net_auc = net_auc[net_auc['CitedYear'] <= eyear]

# Multinomial Link Prediction

In [3]:
panel_data = pd.read_csv(os.path.join(path2data, 'link_prediction_panel_data_1990_2017.csv.gz'))

net_panel = panel_data.merge(net_auc.rename(columns={'CitingCountry':'oCountry', 'CitedCountry':'tCountry','CitedYear':'year'}),
                           how='left', on=['oCountry','tCountry','year'])

net_panel['edge_preference'] = 1 + (net_panel['Zscore'] < 0).astype(int)
net_panel.loc[net_panel['Significant'].isnull(), 'edge_preference'] = 0

# the citation preferences cover 1990-2017, so the panel rows of later years are not used
net_panel = net_panel[(net_panel['year'] >= syear) & (net_panel['year'] <= eyear)].reset_index(drop=True)

dummy_years=['year_'+str(i) for i in range(syear + 1, eyear + 1)]

net_panel_with_dummies = pd.get_dummies(net_panel, columns=['year'], drop_first=True)
net_panel_with_dummies[dummy_years] = net_panel_with_dummies[dummy_years].astype(int)

v1=['oGDP_PCAP','tGDP_PCAP', 'oPop','tPop']
v2=['PhysicalDistance','SameContinent']
v3=['comlang_off','TopicSimilarity']
v4=['cumulative_laws']
v5=['CollabStrength']
v6=['oTopFrac','tTopFrac']

all_model_features = v1 + v2 + v3 + v4 + v5 + v6

# continuous variables are standardized once over the 1990-2017 panel, so all models share the same scale
for c in all_model_features:
    if net_panel_with_dummies[c].nunique() > 2:
        net_panel_with_dummies[c] = (net_panel_with_dummies[c] - net_panel_with_dummies[c].mean())/net_panel_with_dummies[c].std()

exog_var_sets = [v1+dummy_years,v2,v3,v4,v5,v6]
dep_var = "edge_preference"

Nmodels = len(exog_var_sets)

fit_models = []
for icol in range(Nmodels):
    exog_vars = list(chain.from_iterable(exog_var_sets[:(icol+1)]))
    X = net_panel_with_dummies[exog_vars + [dep_var]].dropna().reset_index(drop=True)

    Y = X[dep_var]
    X = sm.add_constant(X[exog_vars])

    model = sm.MNLogit(Y, X).fit()
    fit_models.append(model)

Optimization terminated successfully.
         Current function value: 0.037775
         Iterations 12
Optimization terminated successfully.
         Current function value: 0.038144
         Iterations 12
Optimization terminated successfully.
         Current function value: 0.036957
         Iterations 13
Optimization terminated successfully.
         Current function value: 0.036874
         Iterations 13
Optimization terminated successfully.
         Current function value: 0.035260
         Iterations 12
Optimization terminated successfully.
         Current function value: 0.033921
         Iterations 12


In [4]:
namedict = {'const':'Intercept', 'oTopFrac':'Origin top journal fraction', 'tTopFrac':'Target top journal fraction',
'oGDP_PCAP':'Log origin GDP per capita', 'oPop':'Log origin population','tPop':'Log target population',
           'tGDP_PCAP':'Log target GDP per capita', 'PhysicalDistance': 'Physical distance', 'TopicSimilarity':'Field similarity',
            'comlang_off':'Same official language', 'CollabStrength':'Log collaboration strength','SameContinent':'Same continent',
           'cumulative_laws':'Bilateral research agreements'}
years={i: i[-4:] for i in dummy_years}
merged_namedict = years.copy()
merged_namedict.update(namedict)

# the LaTeX tables are saved in the tables folder
path2tables = os.path.join(os.path.dirname(os.path.abspath(path2sourcedata)), 'tables')
os.makedirs(path2tables, exist_ok=True)

make_multinomial_latex_table(fit_models, exog_var_sets, dep_var, merged_namedict,
                             caption_text="Fixed-effect multinomial logit regression after removing country self-citations.",
                             label="table:multinomial_remove_self", fname=os.path.join(path2tables, "TableS11.tex"))

{\tiny
    \begin{longtable}{p{0.2\linewidth}p{0.12\linewidth}p{0.12\linewidth}p{0.12\linewidth}p{0.12\linewidth}p{0.12\linewidth}p{0.12\linewidth}}\caption{\textbf{Fixed-effect multinomial logit regression after removing country self-citations.} Model coefficients (log-odds of a positive or negative citation preference relative to no preference) with 95\% confidence intervals in parentheses, followed by the standard error (S.E.) and the exact $p$-value. $p$-values are from two-sided Wald $z$-tests of each coefficient, without adjustment for multiple comparisons. The LLR $\chi^2$ is a likelihood-ratio test against the intercept-only model.} 
      \label{table:multinomial_remove_self} \\
      \hline \hline \\
    \multicolumn{7}{c}{\textbf{Dependent variable: Citation preference}} \\ \hline 
     & \multicolumn{6}{c}{Model}  \\\cline{2-7}& (1)& (2)& (3)& (4)& (5)& (6)\\[0.8ex]
    \hline
    \endfirsthead\multicolumn{2}{c}%
      {{\tablename\ \thetable{} -- continued from previous pa

'{\\tiny\n    \\begin{longtable}{p{0.2\\linewidth}p{0.12\\linewidth}p{0.12\\linewidth}p{0.12\\linewidth}p{0.12\\linewidth}p{0.12\\linewidth}p{0.12\\linewidth}}\\caption{\\textbf{Fixed-effect multinomial logit regression after removing country self-citations.} Model coefficients (log-odds of a positive or negative citation preference relative to no preference) with 95\\% confidence intervals in parentheses, followed by the standard error (S.E.) and the exact $p$-value. $p$-values are from two-sided Wald $z$-tests of each coefficient, without adjustment for multiple comparisons. The LLR $\\chi^2$ is a likelihood-ratio test against the intercept-only model.} \n      \\label{table:multinomial_remove_self} \\\\\n      \\hline \\hline \\\\\n    \\multicolumn{7}{c}{\\textbf{Dependent variable: Citation preference}} \\\\ \\hline \n     & \\multicolumn{6}{c}{Model}  \\\\\\cline{2-7}& (1)& (2)& (3)& (4)& (5)& (6)\\\\[0.8ex]\n    \\hline\n    \\endfirsthead\\multicolumn{2}{c}%\n      {{\\tablenam

The same table split into two tables that each fit on one page: TableS11a (positive citation preference) and TableS11b (negative citation preference and the model statistics), written to `tables/TableS11_split.tex`.

In [5]:
make_multinomial_latex_tables_split(fit_models, exog_var_sets, dep_var, merged_namedict,
                                    caption_text="Fixed-effect multinomial logit regression after removing country self-citations.",
                                    label="table:multinomial_remove_self", fname=os.path.join(path2tables, "TableS11_split.tex"))

\refstepcounter{table}\label{table:multinomial_remove_self}\addtocounter{table}{-1}%
\begin{table}[H]
\let\origthetable\thetable\renewcommand{\thetable}{\origthetable a}
\caption{\textbf{Fixed-effect multinomial logit regression after removing country self-citations. (a) Positive citation preference.} Model coefficients (log-odds of a positive citation preference relative to no preference) with 95\% confidence intervals in parentheses, followed by the standard error (S.E.) and the exact $p$-value. $p$-values are from two-sided Wald $z$-tests of each coefficient, without adjustment for multiple comparisons. The coefficients for the negative citation preference and the model statistics are in Table \ref{table:multinomial_remove_self:negative}.}
\label{table:multinomial_remove_self:positive}
\centering
{\tiny
\begin{tabular}{p{0.2\linewidth}p{0.11\linewidth}p{0.11\linewidth}p{0.11\linewidth}p{0.11\linewidth}p{0.11\linewidth}p{0.11\linewidth}}
\hline \hline \\[-1.8ex]
\multicolumn{7}{c}{\t

'\\refstepcounter{table}\\label{table:multinomial_remove_self}\\addtocounter{table}{-1}%\n\\begin{table}[H]\n\\let\\origthetable\\thetable\\renewcommand{\\thetable}{\\origthetable a}\n\\caption{\\textbf{Fixed-effect multinomial logit regression after removing country self-citations. (a) Positive citation preference.} Model coefficients (log-odds of a positive citation preference relative to no preference) with 95\\% confidence intervals in parentheses, followed by the standard error (S.E.) and the exact $p$-value. $p$-values are from two-sided Wald $z$-tests of each coefficient, without adjustment for multiple comparisons. The coefficients for the negative citation preference and the model statistics are in Table \\ref{table:multinomial_remove_self:negative}.}\n\\label{table:multinomial_remove_self:positive}\n\\centering\n{\\tiny\n\\begin{tabular}{p{0.2\\linewidth}p{0.11\\linewidth}p{0.11\\linewidth}p{0.11\\linewidth}p{0.11\\linewidth}p{0.11\\linewidth}p{0.11\\linewidth}}\n\\hline \\hl

## Source Data for Supplementary Table 11

The values reported in the table, written to `Source Data/SourceData_11_TableS11.xlsx`.

In [6]:
title = 'Supplementary Table 11: Fixed-effect multinomial logit regression after removing country self-citations.'
description = ('Coefficients of the multinomial logit models (1)-(6) with temporal fixed-effects predicting a positive '
               'or negative citation preference compared to no preference (baseline), with the citation preferences '
               'recomputed after removing country self-citations. Continuous variables are standardized. The year fixed '
               'effects are included in all models but not shown.')

coefs = []
fits = []
for imodel, model in enumerate(fit_models):
    conf = model.conf_int()
    for icat, outcome in [(1, 'Positive'), (2, 'Negative')]:
        df = pd.DataFrame({'Coefficient':model.params[icat-1], 'SE':model.bse[icat-1], 'p-value':model.pvalues[icat-1]})
        df['CI lower (95%)'] = conf.loc[str(icat)]['lower']
        df['CI upper (95%)'] = conf.loc[str(icat)]['upper']
        df = df[~df.index.str.startswith('year_')]
        df.insert(0, 'Variable', df.index.map(lambda v: merged_namedict.get(v, v)))
        df.insert(0, 'Citation preference', outcome)
        df.insert(0, 'Model', imodel + 1)
        coefs.append(df[['Model', 'Citation preference', 'Variable', 'Coefficient', 'CI lower (95%)', 'CI upper (95%)',
                         'SE', 'p-value']])
    fits.append([imodel + 1, int(model.nobs), model.prsquared, model.llf, model.llr, model.df_model, model.llr_pvalue])

coefs = pd.concat(coefs, ignore_index=True)
fits = pd.DataFrame(fits, columns=['Model', 'Observations', 'Pseudo R2', 'Log likelihood', 'LLR chi2', 'd.f.',
                                   'LLR p-value'])

tables = [('Coefficients', 'Coefficients of each model for the positive and negative citation preference.', coefs),
          ('Model fit', 'Observations and goodness of fit of each model.', fits)]

write_source_data('SourceData_11_TableS11.xlsx', title, description, tables)

wrote /Users/psp2nq/Documents/NationalBiasCode/Source Data/SourceData_11_TableS11.xlsx
wrote /Users/psp2nq/Documents/NationalBiasCode/Source Data/README.txt
